In [ ]:
import sys
import subprocess

def ensure_installed(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

ensure_installed(['sentence-transformers', 'datasets', 'torch', 'scikit-learn', 'pandas', 'numpy'])


In [ ]:
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, classification_report

if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print({'selected_device': device})


In [ ]:
dataset = load_dataset('dair-ai/emotion', split='test')
class_names = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

print({'split': 'test', 'num_rows': len(dataset)})
print(dataset[:3])


In [ ]:
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
model = SentenceTransformer(model_name, device=device)

prototype_map = {
    'sadness': 'This text expresses sadness, unhappiness, or grief.',
    'joy': 'This text expresses joy, happiness, or delight.',
    'love': 'This text expresses love, affection, or deep caring.',
    'anger': 'This text expresses anger, annoyance, or frustration.',
    'fear': 'This text expresses fear, worry, or anxiety.',
    'surprise': 'This text expresses surprise, amazement, or astonishment.'
}

prototype_texts = [prototype_map[name] for name in class_names]
label_to_id = {name: i for i, name in enumerate(class_names)}

prototype_embeddings = model.encode(
    prototype_texts,
    batch_size=16,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=False
)

print({'model_name': model_name, 'class_names': class_names})
print({'prototype_examples': prototype_map})


In [ ]:
texts = dataset['text']
true_ids = dataset['label']

text_embeddings = model.encode(
    texts,
    batch_size=64,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

similarity_matrix = text_embeddings @ prototype_embeddings.T
pred_ids = similarity_matrix.argmax(dim=1).detach().cpu().numpy().tolist()
pred_labels = [class_names[i] for i in pred_ids]
true_labels = [class_names[i] for i in true_ids]
max_scores = similarity_matrix.max(dim=1).values.detach().cpu().numpy()

results_df = pd.DataFrame({
    'text': texts,
    'true_label': true_labels,
    'predicted_label': pred_labels,
    'similarity_score': max_scores
})

print(results_df.head(10).to_dict(orient='records'))


In [ ]:
accuracy = accuracy_score(true_ids, pred_ids)
report = classification_report(true_ids, pred_ids, target_names=class_names, digits=4)

print({
    'model_name': model_name,
    'dataset': 'dair-ai/emotion',
    'split': 'test',
    'num_examples': len(dataset),
    'device': device,
    'accuracy': round(float(accuracy), 6)
})
print(report)


In [ ]:
sample_n = 8
print(results_df.head(sample_n).to_string(index=False))
